# build dataframe

In [68]:
import json
import re
import random
import numpy as np
from pathlib import Path
from time import perf_counter
import tiktoken
import pandas as pd
from spellchecker import SpellChecker
from sklearn.feature_extraction.text import TfidfVectorizer
from bertopic import BERTopic
from sklearn.decomposition import NMF
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN



pd.set_option("display.max_colwidth", None)

In [2]:
# load dataset

def load_json(path):
    with Path(path).open("r", encoding="utf-8") as file:
        return json.load(file)


processed_path = Path(
    "C:/Users/heike/Desktop/Stackfuel/Portfolio/llm-sustainability-analysis/01_data/02_processed/sharegpt_english.json"
)

english_data = load_json(processed_path)

print(f"Loaded conversations: {len(english_data)}")

Loaded conversations: 58921


# structured features

In [9]:
# conversation ids

conversation_ids = [
    item.get("id")
    for item in english_data
    if isinstance(item, dict) and item.get("id") is not None
]

print(f"Conversation IDs: {len(conversation_ids)}")
print(f"Unique Conversation IDs: {len(set(conversation_ids))}")
print(f"Duplicates: {len(conversation_ids) - len(set(conversation_ids))}")
    

Conversation IDs: 58921
Unique Conversation IDs: 58921
Duplicates: 0


In [10]:
# build dataframe

rows = []

encoding = tiktoken.get_encoding("cl100k_base")
spell = SpellChecker(language="en")


# =========================
# helper functions
# =========================

def get_first_prompt_response_pair(tree):
    conversations = tree.get("conversations", [])

    for i in range(len(conversations) - 1):
        current = conversations[i]
        next_msg = conversations[i + 1]

        if current.get("from") == "human" and next_msg.get("from") == "gpt":
            return current.get("value", ""), next_msg.get("value", "")

    return "", ""


def get_words(prompt):
    return re.findall(r"[A-Za-z']+", prompt.lower())


# =========================
# token-based features
# =========================

def count_tokens(text):
    return len(
        encoding.encode(
            text,
            disallowed_special=()
        )
    )


def count_user_prompts(tree):
    return sum(
        message.get("from") == "human"
        for message in tree.get("conversations", [])
    )


def count_total_user_tokens(tree):
    total = 0

    for message in tree.get("conversations", []):
        if message.get("from") == "human":
            total += count_tokens(message.get("value", ""))

    return total


def count_total_assistant_tokens(tree):
    total = 0

    for message in tree.get("conversations", []):
        if message.get("from") == "gpt":
            total += count_tokens(message.get("value", ""))

    return total


# =========================
# prompt design features
# =========================

def has_role_instruction(prompt):
    prompt = prompt.lower()

    role_patterns = [
        "act as",
        "you are",
        "you're",
        "pretend you are",
        "imagine you are",
        "take the role",
        "role of",
    ]

    return any(pattern in prompt for pattern in role_patterns)


def has_audience_or_level_instruction(prompt):
    prompt = prompt.lower()

    patterns = [
        "explain it to me like",
        "explain like i am",
        "explain like i'm",
        "eli5",
        "like i am 5",
        "like i'm 5",
        "like i am 10",
        "like i'm 10",
        "for beginners",
        "to a beginner",
        "in simple terms",
        "plain english",
        "non-technical",
        "for a child",
        "for a high school student",
    ]

    return any(pattern in prompt for pattern in patterns)


def has_format_instruction(prompt):
    prompt = prompt.lower()

    format_keywords = [
        "bullet point",
        "bullet points",
        "table",
        "json",
        "csv",
        "list",
        "markdown",
        "format",
        "section",
        "sections",
        "outline",
        "step by step",
        "numbered",
    ]

    return any(keyword in prompt for keyword in format_keywords)


def count_questions(prompt):
    return prompt.count("?")


def prompt_style(prompt):
    prompt_lower = prompt.lower().strip()

    instruction_verbs = [
        "write", "summarize", "explain", "create", "generate", "translate",
        "list", "compare", "analyze", "give", "make", "draft", "compose",
        "rewrite", "classify", "extract", "find", "calculate",
    ]

    if "?" in prompt_lower:
        return "question"

    if any(prompt_lower.startswith(verb) for verb in instruction_verbs):
        return "instruction"

    return "other"


# =========================
# orthographic features
# =========================

def orthographic_error_rate(prompt):
    words = get_words(prompt)

    if not words:
        return 0

    unknown_words = spell.unknown(words)

    return len(unknown_words) / len(words)


# =========================
# task type
# =========================

def classify_task_type(prompt):
    prompt = prompt.lower()

    if any(word in prompt for word in ["python", "java", "javascript", "typescript", "sql", "html", "css", "api",
                                       "debug", "error", "exception", "bash", "terminal", "command", "linux", "c++", "cpp"]):
        return "coding"

    if any(word in prompt for word in ["email", "subject line", "reply to", "newsletter"]):
        return "email_writing"

    if any(word in prompt for word in ["summarize", "summary", "tl;dr", "key points"]):
        return "summarization"

    if any(word in prompt for word in ["translate", "translation", "in english", "into english", "into spanish", "into french"]):
        return "translation"

    if any(word in prompt for word in ["explain", "what is", "how does", "why does", "difference between", "simple terms"]):
        return "explanation"

    if any(word in prompt for word in ["write", "draft", "compose", "story", "poem", "article", "script", "blog post"]):
        return "writing_generation"

    if any(word in prompt for word in ["idea", "ideas", "brainstorm", "suggest", "recommend"]):
        return "brainstorming"

    if any(word in prompt for word in ["act as", "pretend", "roleplay", "you are now", "simulate"]):
        return "roleplay"

    return "general_assistance"


# =========================
# build conversation-level rows
# =========================

for tree in english_data:

    first_prompt, first_response = get_first_prompt_response_pair(tree)

    total_turns = len(tree.get("conversations", []))
    user_prompts = count_user_prompts(tree)

    total_user_tokens = count_total_user_tokens(tree)
    total_assistant_tokens = count_total_assistant_tokens(tree)
    total_tokens = total_user_tokens + total_assistant_tokens
    log_total_tokens = np.log1p(total_tokens)

    follow_up_prompts = max(0, user_prompts - 1)
    needs_follow_up = follow_up_prompts > 0

    rows.append({
        "conversation_id": tree.get("id"),

        # CORE PAIR
        "first_prompt": first_prompt,
        "first_response": first_response,

        # TOKENS
        "first_prompt_tokens": count_tokens(first_prompt),
        "first_response_tokens": count_tokens(first_response),

        # STRUCTURE
        "total_turns": total_turns,
        "interaction_rounds": total_turns / 2,

        # COST SIGNALS
        "total_user_tokens": total_user_tokens,
        "total_assistant_tokens": total_assistant_tokens,
        "total_tokens": total_tokens,
        "log_total_tokens": log_total_tokens,

        # BEHAVIOR
        "follow_up_prompts": follow_up_prompts,
        "needs_follow_up": needs_follow_up,

        # PROMPT FEATURES
        "has_role_instruction": has_role_instruction(first_prompt),
        "has_audience_or_level_instruction": has_audience_or_level_instruction(first_prompt),
        "has_format_instruction": has_format_instruction(first_prompt),
        "question_count": count_questions(first_prompt),
        "prompt_style": prompt_style(first_prompt),

        # TASK
        "task_type": classify_task_type(first_prompt),

        # ORTHOGRAPHIC
        "orthographic_error_rate": orthographic_error_rate(first_prompt),
    })


df = pd.DataFrame(rows)

boolean_columns = [
    "has_role_instruction",
    "has_audience_or_level_instruction",
    "has_format_instruction",
]

df[boolean_columns] = df[boolean_columns].astype(int)
df["needs_follow_up"] = df["needs_follow_up"].astype(int)
df["question_count"] = df["question_count"].astype(int)

df.head()

# sanity check
len(df), len(english_data)

(58921, 58921)

# check NaNs

In [33]:
# check NaNs in dataframe
#print("n with NaNs:", df.isna().sum())     # --- n=1245 NaNs in first_prompt, first_response

#df = df[
#    (df["first_prompt"] != "") &
#    (df["first_response"] != "")
#]

# probably: broken conversations, unpaired chats, incomplete samples; drop 1245 empty conversations

#print("n without NaNs:", df.isna().sum())

# embedding

In [16]:
df = df[df["first_prompt"].str.len() > 0].reset_index(drop=True)

In [7]:
def is_valid_prompt(text):
    if not text or len(text) < 10:
        return False

    noise_markers = [
        "root@",
        "#",
        "-----",
        "Enter pass phrase",
        "Select an option"
    ]

    return not any(m in text for m in noise_markers)


def get_first_user_prompt(tree):
    for msg in tree.get("conversations", []):
        if msg.get("from") != "human":
            continue

        text = msg.get("value", "")

        if is_valid_prompt(text):
            return text

    return ""

In [44]:

embedder = SentenceTransformer("all-MiniLM-L6-v2")

df = df.copy().reset_index(drop=True)

prompts = df["first_prompt"].fillna("").astype(str).to_numpy()

def embed_chunked(texts, model, chunk_size=5000, batch_size=64):
    all_emb = []

    for i in tqdm(range(0, len(texts), chunk_size)):
        chunk = texts[i:i+chunk_size]

        emb = model.encode(
            chunk,
            batch_size=batch_size,
            show_progress_bar=False
        )

        all_emb.append(emb)

    return np.concatenate(all_emb, axis=0)

X_emb = embed_chunked(prompts, embedder)

assert len(X_emb) == len(df)


100%|██████████| 12/12 [25:52<00:00, 129.40s/it]


In [45]:
assert len(X_emb) == len(df)

In [46]:
print(len(X_emb))
print(len(df))

57529
57529


In [47]:
assert df["embedding"].iloc[0].shape[0] == X_emb.shape[1]
assert len(df) == len(X_emb)

In [49]:
df = df.reset_index(drop=True)

np.save(
    r"C:/Users/heike/Desktop/Stackfuel/Portfolio/llm-sustainability-analysis/01_data/03_features/embeddings.npy",
    X_emb
)

# topic modelling

In [64]:
vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5
)

In [66]:
umap_model = UMAP(
    n_neighbors=30,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

In [ ]:
hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    min_samples=10,
    metric="euclidean",
    prediction_data=True
)

In [69]:
from bertopic import BERTopic

topic_model = BERTopic(
    embedding_model=embedder,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    calculate_probabilities=True,
    min_topic_size=50,
    nr_topics="auto"
)

topics, probs = topic_model.fit_transform(df["first_prompt"].tolist())

In [70]:
df["topic"] = topics
df["topic_prob"] = probs.max(axis=1)

In [71]:
# check results topic modelling

topic_info = topic_model.get_topic_info()
topic_info.head(100)
topic_model.get_topic(0)

[('write', np.float64(0.012289440072947312)),
 ('chatgpt', np.float64(0.011927656415399604)),
 ('use', np.float64(0.00991365010981196)),
 ('data', np.float64(0.00933437300146266)),
 ('like', np.float64(0.009258194824668456)),
 ('make', np.float64(0.008973957269703082)),
 ('want', np.float64(0.008883479190902044)),
 ('using', np.float64(0.008870873596156207)),
 ('text', np.float64(0.008534125352809506)),
 ('prompt', np.float64(0.008408149628360666))]

In [73]:
topic_model.get_topic(5)

[('patient', np.float64(0.045463315422550755)),
 ('mg', np.float64(0.03938565572726781)),
 ('blood', np.float64(0.03630508977179095)),
 ('ct', np.float64(0.03276067647531616)),
 ('patients', np.float64(0.030297626755312876)),
 ('disease', np.float64(0.026675102547735377)),
 ('00', np.float64(0.02516404264326496)),
 ('cancer', np.float64(0.022834366634577295)),
 ('cells', np.float64(0.021603752430109327)),
 ('old', np.float64(0.021031349596193624))]

In [54]:
topic_model.reduce_topics(df["first_prompt"].tolist(), nr_topics=50)

In [80]:
topic_info = topic_model.get_topic_info()

df = df.merge(
    topic_info[["Topic", "Name"]],
    left_on="topic",
    right_on="Topic",
    how="left"
)

df = df.rename(columns={"Name": "topic_label"}).drop(columns=["Topic"])


In [83]:
topic_counts = (
    df[df["topic"] != -1]["topic"]
    .value_counts()
    .reset_index()
)

topic_counts.columns = ["topic", "count"]

top5 = topic_counts.head(20)

top5 = top5.merge(
    topic_model.get_topic_info()[["Topic", "Name"]],
    left_on="topic",
    right_on="Topic",
    how="left"
)

top5

,topic,count,Topic,Name
0,0,22662,0,0_write_chatgpt_use_data
1,1,589,1,1_trip_visit_travel_day
2,2,431,2,2_000_minutes_total_did
3,3,337,3,3_recipe_chicken_add_ingredients
4,4,262,4,4_git_github_commit_branch
5,5,219,5,5_patient_mg_blood_ct
6,6,204,6,6_mathematical_equation_math_2x
7,7,187,7,7_1i_1can_1what_help
8,8,187,8,8_quantum_computing_simple terms_terms
9,9,179,9,9_excel_sheet_column_cell


# targets

In [84]:
# -------------------------
# leakage-safe targets
# -------------------------

def count_tokens(text):
    return len(encoding.encode(text, disallowed_special=()))

df["target_cost"] = df["first_response"].apply(count_tokens)


df["target_success"] = ~df["first_response"].str.contains(r"\?")

# save

In [85]:
base_path = Path(
    "C:/Users/heike/Desktop/Stackfuel/Portfolio/llm-sustainability-analysis/01_data/03_features/conversation_features.csv"
)

base_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(base_path, index=False)